In [0]:
MY_ID = "preetam"
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"

spark.sql(f"CREATE CATALOG IF NOT EXISTS workspace")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.capstone_{MY_ID}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.capstone_{MY_ID}.raw")

DataFrame[]

In [0]:
# STEP 1 — Run the generator exactly as given

import pyspark.sql.functions as F

OPT = {'header': True, 'ignoreLeadingWhiteSpace': False, 'ignoreTrailingWhiteSpace': False}
def wr(d, p): d.write.mode('overwrite').options(**OPT).csv(f'{VOL}/raw/{p}')

SB = [(0,'MATH','Engineering Mathematics',0.50),(1,'PHYS','Applied Physics',0.30),
      (2,'PROG','Programming in C',0.55),(3,'DBMS','Database Systems',0.35),
      (4,'ENGL','Technical English',0.10),(5,'WORK','Workshop Practice',0.05)]
sub = spark.createDataFrame(SB, 'sj int,subject_code string,subject_name string,beta double')
wr(sub.select('subject_code','subject_name'), 'subjects')

stu = spark.range(1200).selectExpr("id","format_string('S%04d',id+1) student_id",
    "if(id%3=0,'CSE',if(id%3=1,'ECE','MECH')) branch",
    "format_string('SEC%d',id%4+1) section",
    "round(0.50+rand(84)*0.48,3) diligence","if(id%8=3,12.0,0.0) penalty")
wr(stu.selectExpr("student_id","branch","section"), 'students')

m0 = (stu.crossJoin(spark.range(1,4).withColumnRenamed('id','semester')).crossJoin(sub)
    .selectExpr("student_id","semester","subject_code","beta","penalty",
       "pmod((id*18+(semester-1)*6+sj)*7919,21600) kk","if(subject_code='WORK',30,45) hd",
       "round(least(99.0,greatest(35.0,diligence*100+randn(85)*6)),1) ap","randn(86)*9 nz"))
T = "least(100.0,greatest(0.0,round(52+beta*(ap-75)+nz-if(semester=2,penalty,0.0))))"
m1 = m0.selectExpr("student_id","semester","subject_code","kk","hd",
    "cast(round(hd*ap/100) as int) at", f"cast({T} as int) tm")
mk = m1.selectExpr("student_id","semester","subject_code","kk",
    "cast(if(kk>=650 and kk<740,0,hd) as string) classes_held",
    "cast(if(kk>=300 and kk<500,hd+5,at) as string) classes_attended",
    "if(kk<300,'AB',cast(cast(round(tm*0.4) as int) as string)) internal_marks",
    "cast(if(kk>=500 and kk<650,70,tm-cast(round(tm*0.4) as int)) as string) external_marks")
SC = "element_at(array('MATH','PHYS','PROG','DBMS','ENGL','WORK'),cast(id%6+1 as int))"
op = spark.range(120).selectExpr("format_string('S9%03d',id+1) student_id",
    "cast(id%3+1 as bigint) semester", f"{SC} subject_code","cast(-1 as bigint) kk",
    "'45' classes_held","'38' classes_attended","'22' internal_marks","'34' external_marks")
wr(mk.unionByName(mk.filter('kk>=740 and kk<1220')).unionByName(op).drop('kk'), 'marks')

In [0]:
print(spark.read.csv(f'{VOL}/raw/marks', header=True).count())   # expect 22200

22200
